## 1. Dimension: Hazard

### 1.1 Indicator: Drought Vulnerability

In [1]:
import pandas as pd
import numpy as np

In [2]:
drought = pd.read_csv("../data/raw/hazard/drought_vulnerability.csv")

In [3]:
drought.head()

,_id,cve_concatenada,cve_rha,nom_rha,probabilidad,t_prob
0,1,1001,8,Lerma Santiago Pacifico,79.37,Alta
1,2,1002,8,Lerma Santiago Pacifico,66.02,Alta
2,3,1003,8,Lerma Santiago Pacifico,1.84,Muy Baja
3,4,1004,8,Lerma Santiago Pacifico,54.00,Media
4,5,1005,8,Lerma Santiago Pacifico,27.57,Baja


In [4]:
drought.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2463 entries, 0 to 2462
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   _id              2463 non-null   int64  
 1   cve_concatenada  2463 non-null   int64  
 2   cve_rha          2463 non-null   int64  
 3   nom_rha          2463 non-null   object 
 4   probabilidad     2463 non-null   float64
 5   t_prob           2463 non-null   object 
dtypes: float64(1), int64(3), object(2)
memory usage: 115.6+ KB


In [6]:
drought.shape

(2463, 6)

In [7]:
state_lookup = pd.DataFrame({
    "state_code": [
        "01","02","03","04","05","06","07","08",
        "09","10","11","12","13","14","15","16",
        "17","18","19","20","21","22","23","24",
        "25","26","27","28","29","30","31","32"
    ],
    "state_name": [
        "Aguascalientes",
        "Baja California",
        "Baja California Sur",
        "Campeche",
        "Coahuila de Zaragoza",
        "Colima",
        "Chiapas",
        "Chihuahua",
        "Ciudad de México",
        "Durango",
        "Guanajuato",
        "Guerrero",
        "Hidalgo",
        "Jalisco",
        "México",
        "Michoacán de Ocampo",
        "Morelos",
        "Nayarit",
        "Nuevo León",
        "Oaxaca",
        "Puebla",
        "Querétaro",
        "Quintana Roo",
        "San Luis Potosí",
        "Sinaloa",
        "Sonora",
        "Tabasco",
        "Tamaulipas",
        "Tlaxcala",
        "Veracruz de Ignacio de la Llave",
        "Yucatán",
        "Zacatecas"
    ]
})

state_lookup

,state_code,state_name
0,01,Aguascalientes
1,02,Baja California
2,03,Baja California Sur
3,04,Campeche
4,05,Coahuila de Zaragoza
5,06,Colima
6,07,Chiapas
7,08,Chihuahua
8,09,Ciudad de México
9,10,Durango


In [8]:
state_lookup.to_csv(
    "../data/state_codes.csv",
    index=False
)

In [9]:
drought = drought[["cve_concatenada", "probabilidad"]].copy()

In [10]:
drought.head()

,cve_concatenada,probabilidad
0,1001,79.37
1,1002,66.02
2,1003,1.84
3,1004,54.00
4,1005,27.57


In [11]:
drought["state_code"] = (
    drought["cve_concatenada"]
    .astype(str)
    .str.zfill(5)
    .str[:2]
)

In [12]:
drought.head()

,cve_concatenada,probabilidad,state_code
0,1001,79.37,01
1,1002,66.02,01
2,1003,1.84,01
3,1004,54.00,01
4,1005,27.57,01


In [13]:
hazard_drought = (
    drought
    .groupby("state_code", as_index=False)
    .agg(
        drought_susceptibility=("probabilidad", "mean")
    )
)

In [17]:
state_lookup = pd.read_csv(
    "../data/state_codes.csv",
    dtype={"state_code": str}
)

In [18]:
hazard_drought = hazard_drought.merge(
    state_lookup,
    on="state_code",
    how="left"
)

In [21]:
hazard_drought.head()

,state_code,state_name,drought_susceptibility
0,01,Aguascalientes,52.530000
1,02,Baja California,64.800000
2,03,Baja California Sur,31.106000
3,04,Campeche,14.763636
4,05,Coahuila de Zaragoza,31.983684


In [20]:
hazard_drought = hazard_drought[
    [
        "state_code",
        "state_name",
        "drought_susceptibility"
    ]
]

In [22]:
hazard_drought.isna().sum()

state_code                0
state_name                0
drought_susceptibility    0
dtype: int64

In [23]:
hazard_drought["state_code"].nunique()

32

In [24]:
hazard_drought.to_csv(
    "../data/processed/hazard_drought.csv",
    index=False
)

### 1.2 Indicator: Climate vulnerability

In [261]:
climate_vulnerability = pd.read_excel("../data/raw/hazard/Indicadores_municipales.xlsx")

In [262]:
climate_vulnerability.head(10)

,CVEGEO,CVE_ENT,CVE_MUN,NOM_ENT,NOM_MUN,POBTOT,POBFEM,POBMAS,P_0A11_F,P_0A11_M,...,NOMAGRUP,G_RESILIEN,V_CC,CONT_SLO,CONT_AGUA,PLAGAS,ENF_INTEPI,GP_IF,GP_Trans,GP_Alm
0,1001,1,1,Aguascalientes,Aguascalientes,948990.0,486917.0,462073.0,94197,96889,...,NaN,Muy alto,No,Muy alto,Medio,Medio,Medio,Alto,Muy alto,Muy alto
1,1002,1,2,Aguascalientes,Asientos,51536.0,26275.0,25261.0,6583,6534,...,NaN,Alto,No,Muy alto,Bajo,Medio,Medio,Bajo,Muy bajo,Medio
2,1003,1,3,Aguascalientes,Calvillo,58250.0,29687.0,28563.0,6601,6836,...,NaN,Alto,No,Bajo,Medio,Medio,Medio,Medio,Bajo,Bajo
3,1004,1,4,Aguascalientes,Cosío,17000.0,8708.0,8292.0,2086,2094,...,NaN,Alto,No,Bajo,Muy alto,Medio,Bajo,Muy bajo,Muy bajo,Bajo
4,1005,1,5,Aguascalientes,Jesús María,129929.0,65710.0,64219.0,14956,15584,...,NaN,Muy alto,No,Medio,Muy alto,Medio,Bajo,Medio,Bajo,Alto
5,1006,1,6,Aguascalientes,Pabellón de Arteaga,47646.0,24269.0,23377.0,5564,5794,...,NaN,Muy alto,No,Sin dato,Medio,Medio,Medio,Bajo,Bajo,Bajo
6,1007,1,7,Aguascalientes,Rincón de Romos,57369.0,29268.0,28101.0,6979,7302,...,NaN,Muy alto,No,Medio,Alto,Medio,Bajo,Medio,Muy bajo,Bajo
7,1008,1,8,Aguascalientes,San José de Gracia,9552.0,5020.0,4532.0,1282,1202,...,NaN,Alto,No,Bajo,Medio,Bajo,Bajo,Medio,Muy bajo,Bajo
8,1009,1,9,Aguascalientes,Tepezalá,22485.0,11371.0,11114.0,2753,2801,...,NaN,Alto,Si,Bajo,Bajo,Medio,Bajo,Muy bajo,Muy bajo,Bajo
9,1010,1,10,Aguascalientes,El Llano,20853.0,10407.0,10446.0,2462,2539,...,NaN,Alto,Si,Bajo,Bajo,Medio,Medio,Muy bajo,Muy bajo,Bajo


In [263]:
climate_vulnerability.columns

Index(['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'NOM_ENT', 'NOM_MUN', 'POBTOT',
       'POBFEM', 'POBMAS', 'P_0A11_F', 'P_0A11_M', 'P_12A17_F', 'P_12A17_M',
       'POB65_MAS', 'P_60YMAS_F', 'P_60YMAS_M', 'P3YM_HLI', 'POB_AFRO',
       'PCON_DISC', 'PSINDER', 'VIVTOT', 'AREA', 'DEN_POBKM', 'GP_ONDASCA',
       'GP_CICLNES', 'GP_BAJASTE', 'GP_NEVADAS', 'GP_GRANIZO', 'GP_TORMELE',
       'GP_SEQUIA2', 'GP_INUNDAC', 'SUSCEPLAD', 'GP_SISMICO', 'GP_TSUNAMI',
       'VOLCANES', 'IVS2020', 'GVS2020', 'I_MARGINAC', 'G_MARGINAC', 'IRS2020',
       'GRS2020', 'POB_POR10', 'POB_POR15', 'POB_POR20', 'POB_PER10',
       'POB_PER15', 'POB_PER20', 'POBE_POR10', 'POBE_POR15', 'POBE_POR20',
       'POBE_PER10', 'POBE_PER15', 'POBE_PER20', 'GASOLINERA', 'HOTELES',
       'BANCOS', 'SUPERMERCA', 'CC_HIDRO', 'D_GEO', 'D_HIDRO', 'D_QUI',
       'E_GEO', 'E_HIDRO', 'E_QUI', 'E_SANI', 'DECLARATOR', 'TITULOREGL',
       'NOMAGRUP', 'G_RESILIEN', 'V_CC', 'CONT_SLO', 'CONT_AGUA', 'PLAGAS',
       'ENF_INTEPI', 'GP_IF', 

In [264]:
climate_vulnerability["GP_SEQUIA2"].value_counts(dropna=False)

GP_SEQUIA2
Bajo        1082
Medio        779
Muy bajo     424
Alto         175
Muy alto       9
Name: count, dtype: int64

In [265]:
hazard_scale = {
    "Muy bajo": 1,
    "Bajo": 2,
    "Medio": 3,
    "Alto": 4,
    "Muy alto": 5
}

In [266]:
climate_vulnerability["flood_hazard"] = climate_vulnerability["GP_INUNDAC"].map(hazard_scale)

climate_vulnerability["cyclone_hazard"] = climate_vulnerability["GP_CICLNES"].map(hazard_scale)

climate_vulnerability["drought_hazard"] = climate_vulnerability["GP_SEQUIA2"].map(hazard_scale)

In [268]:
climate_vulnerability[
    [
        "GP_INUNDAC",
        "flood_hazard",
        "GP_CICLNES",
        "cyclone_hazard",
        "GP_SEQUIA2",
        "drought_hazard"
    ]
].head(10)

,GP_INUNDAC,flood_hazard,GP_CICLNES,cyclone_hazard,GP_SEQUIA2,drought_hazard
0,Medio,3,Muy bajo,1,Bajo,2
1,Medio,3,Muy bajo,1,Bajo,2
2,Muy bajo,1,Muy bajo,1,Bajo,2
3,Bajo,2,Muy bajo,1,Bajo,2
4,Bajo,2,Muy bajo,1,Bajo,2
5,Alto,4,Muy bajo,1,Bajo,2
6,Medio,3,Muy bajo,1,Bajo,2
7,Bajo,2,Muy bajo,1,Bajo,2
8,Bajo,2,Muy bajo,1,Bajo,2
9,Alto,4,Muy bajo,1,Bajo,2


In [269]:
climate_vulnerability[
    [
        "flood_hazard",
        "cyclone_hazard",
        "drought_hazard"
    ]
].isna().sum()

flood_hazard      0
cyclone_hazard    0
drought_hazard    0
dtype: int64

In [274]:
def weighted_average(group, value_column):
    return (
        (group[value_column] * group["POBTOT"]).sum()
        / group["POBTOT"].sum()
    )

In [275]:
hazard = climate_vulnerability.groupby("NOM_ENT").apply(
    lambda x: pd.Series({
        "flood_hazard": weighted_average(x, "flood_hazard"),
        "cyclone_hazard": weighted_average(x, "cyclone_hazard"),
        "drought_hazard": weighted_average(x, "drought_hazard")
    })
).reset_index()

/var/folders/s6/mc1_mgvs57q3pb8ltrj2f_mm0000gn/T/ipykernel_77958/2121407802.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hazard = climate_vulnerability.groupby("NOM_ENT").apply(


In [276]:
hazard[
    [
        "flood_hazard",
        "cyclone_hazard",
        "drought_hazard"
    ]
] = hazard[
    [
        "flood_hazard",
        "cyclone_hazard",
        "drought_hazard"
    ]
].round(2)

In [279]:
hazard.tail(10)

,NOM_ENT,flood_hazard,cyclone_hazard,drought_hazard
22,Quintana Roo,4.13,4.36,3.00
23,San Luis Potosí,4.06,1.07,2.06
24,Sinaloa,4.23,3.83,3.70
25,Sonora,3.12,2.39,3.80
26,Tabasco,5.00,1.50,2.94
27,Tamaulipas,4.75,2.74,3.12
28,Tlaxcala,2.53,1.00,2.00
29,Veracruz de Ignacio de la Llave,4.45,2.03,2.43
30,Yucatán,3.28,3.48,3.00
31,Zacatecas,2.85,1.14,2.32


In [280]:
hazard.shape

(32, 4)

In [281]:
hazard = hazard.rename(columns={
    "NOM_ENT": "state_name"
})

In [282]:
hazard = state_lookup.merge(
    hazard,
    on="state_name",
    how="left"
)

In [283]:
hazard.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   state_code      32 non-null     object 
 1   state_name      32 non-null     object 
 2   flood_hazard    32 non-null     float64
 3   cyclone_hazard  32 non-null     float64
 4   drought_hazard  32 non-null     float64
dtypes: float64(3), object(2)
memory usage: 1.4+ KB


In [284]:
hazard.head()

,state_code,state_name,flood_hazard,cyclone_hazard,drought_hazard
0,01,Aguascalientes,2.80,1.00,2.00
1,02,Baja California,3.34,1.49,4.24
2,03,Baja California Sur,2.32,4.98,2.98
3,04,Campeche,4.79,3.12,3.00
4,05,Coahuila de Zaragoza,2.94,1.18,2.96


In [285]:
hazard.to_csv(
    "../data/processed/hazard_indicators.csv",
    index=False
)

## 2. Dimension: Exposure

### 2.1 Indicator: Population Density

In [201]:
population_density = pd.read_excel(
    "../data/raw/exposure/population_density.xlsx"
)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [205]:
population_density.head(10)

,state_name,population_density
0,Aguascalientes,253.9
1,Baja California,52.8
2,Baja California Sur,10.8
3,Campeche,16.1
4,Coahuila de Zaragoza,20.8
5,Colima,130.0
6,Chiapas,75.6
7,Chihuahua,15.1
8,Ciudad de México,6163.3
9,Durango,14.9


In [203]:
population_density.columns = [
    "state_name",
    "population_density"
]

In [204]:
population_density = population_density.iloc[6:].reset_index(drop=True)

In [206]:
population_density = population_density[
    pd.to_numeric(
        population_density["population_density"],
        errors="coerce"
    ).notna()
]

In [209]:
population_density["population_density"] = pd.to_numeric(
    population_density["population_density"]
)

In [210]:
population_density = population_density.merge(
    state_lookup,
    on="state_name",
    how="left"
)

In [211]:
population_density = population_density[
    [
        "state_code",
        "state_name",
        "population_density"
    ]
]

In [212]:
population_density.shape

(32, 3)

In [214]:
population_density.describe()

,population_density
count,32.000000
mean,309.684375
std,1078.690324
min,10.800000
25%,43.400000
50%,67.150000
75%,159.050000
max,6163.300000


In [215]:
population_density.to_csv(
    "../data/processed/exposure_population_density.csv",
    index=False
)

### 2.2 Indicator: Agricultural Land Share 

In [216]:
agricultural_land = pd.read_excel(
    "../data/raw/exposure/land_use.xlsx"
)

In [219]:
agricultural_land.head(20)

,INEGI. Censo Agropecuario 2022,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Cuadro ca2022_02
0,Superficie total y de uso agropecuario y fores...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Datos de octubre de 2021 a septiembre de 2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Superficie en hectáreas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Entidad federativa,Superficie del país,NaN,NaN,Superficie declarada de las unidades de produc...,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,Total,Uso agrícola,NaN,NaN,NaN,"Agostadero, pastos naturales, enmontada, bosqu...",Con otros usos
6,NaN,Total,Superficie con uso o vocación agropecuaria y a...,NaN,NaN,Total,Sembrada u ocupada con cultivos,No sembrada,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Total,En descanso,NaN,NaN
8,NaN,NaN,Total,Cubierta por el censo,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,A,B<=A,C<=B,D=E+I+J,E=F+G,F,G,H<=G,I,J


In [220]:
agricultural_land.columns

Index(['INEGI. Censo Agropecuario 2022', 'Unnamed: 1', 'Unnamed: 2',
       'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7',
       'Unnamed: 8', 'Unnamed: 9', 'Cuadro ca2022_02'],
      dtype='object')

In [221]:
agricultural_land.shape

(46, 11)

In [222]:
agricultural_land = agricultural_land.iloc[10:].reset_index(drop=True)

In [224]:
agricultural_land.iloc[0]

INEGI. Censo Agropecuario 2022    ESTADOS UNIDOS MEXICANOS
Unnamed: 1                                196265063.263785
Unnamed: 2                                  103590781.3122
Unnamed: 3                                  102155122.1538
Unnamed: 4                                   88166735.8114
Unnamed: 5                                   29806776.3228
Unnamed: 6                                   21937852.0501
Unnamed: 7                                    7868924.2727
Unnamed: 8                                     5996840.427
Unnamed: 9                                   57291315.5969
Cuadro ca2022_02                              1068643.8917
Name: 0, dtype: object

In [225]:
agricultural_land = agricultural_land[
    [
        "INEGI. Censo Agropecuario 2022",
        "Unnamed: 1",
        "Unnamed: 2"
    ]
]

In [226]:
agricultural_land.columns = [
    "state_name",
    "state_area",
    "agricultural_area"
]

In [227]:
agricultural_land.head()

,state_name,state_area,agricultural_area
0,ESTADOS UNIDOS MEXICANOS,196265063.263785,103590781.3122
1,Aguascalientes,561569.244381,313698.65
2,Baja California,7319308.004428,2432056.4368
3,Baja California Sur,7202406.148726,1479391.5854
4,Campeche,5750800.732924,3206083.1787


In [233]:
agricultural_land.tail(10)

,state_name,state_area,agricultural_area
22,Quintana Roo,4484126.614547,1586832.7551
23,San Luis Potosí,6113796.241635,2463717.1056
24,Sinaloa,5739107.228936,3009922.1187
25,Sonora,18061018.916969,10222153.6721
26,Tabasco,2473806.494434,1960608.8786
27,Tamaulipas,8027068.138742,5441633.9164
28,Tlaxcala,399663.41957,261975.4887
29,Veracruz de Ignacio de la Llave,7182985.464149,5882840.1249
30,Yucatán,3976098.559598,1789857.3196
31,Zacatecas,7527539.082265,3138664.9263


In [229]:
agricultural_land = agricultural_land[
    agricultural_land["state_name"] != "ESTADOS UNIDOS MEXICANOS"
]

In [230]:
agricultural_land = agricultural_land.dropna(
    subset=["state_area", "agricultural_area"]
)

In [231]:
agricultural_land = agricultural_land.reset_index(drop=True)

In [232]:
agricultural_land.shape

(32, 3)

In [234]:
agricultural_land["agricultural_land_share"] = (
    agricultural_land["agricultural_area"] /
    agricultural_land["state_area"]
) * 100

In [235]:
agricultural_land["agricultural_land_share"] = (
    agricultural_land["agricultural_land_share"]
    .round(2)
)

In [238]:
agricultural_land.describe()

,state_name,state_area,agricultural_area,agricultural_land_share
count,32,32.000000,32.00,32.000000
unique,32,32.000000,32.00,32.000000
top,Aguascalientes,561569.244381,313698.65,55.861081
freq,1,1.000000,1.00,1.000000


In [239]:
agricultural_land["state_area"] = pd.to_numeric(
    agricultural_land["state_area"],
    errors="coerce"
)

agricultural_land["agricultural_area"] = pd.to_numeric(
    agricultural_land["agricultural_area"],
    errors="coerce"
)

In [240]:
agricultural_land["agricultural_land_share"] = (
    agricultural_land["agricultural_area"] /
    agricultural_land["state_area"]
) * 100

agricultural_land["agricultural_land_share"] = (
    agricultural_land["agricultural_land_share"].round(2)
)

In [241]:
agricultural_land.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   state_name               32 non-null     object 
 1   state_area               32 non-null     float64
 2   agricultural_area        32 non-null     float64
 3   agricultural_land_share  32 non-null     float64
dtypes: float64(3), object(1)
memory usage: 1.1+ KB


In [242]:
agricultural_land.describe()

,state_area,agricultural_area,agricultural_land_share
count,3.200000e+01,3.200000e+01,32.000000
mean,6.133283e+06,3.237212e+06,52.718437
std,5.389234e+06,3.183374e+06,12.867016
min,1.494313e+05,5.870161e+04,20.540000
25%,2.414150e+06,1.427225e+06,43.695000
50%,5.805594e+06,2.447887e+06,52.480000
75%,7.380611e+06,4.045299e+06,60.937500
max,2.474125e+07,1.540624e+07,81.900000


In [243]:
agricultural_land = agricultural_land.merge(
    state_lookup,
    on="state_name",
    how="left"
)

In [244]:
agricultural_land = agricultural_land[
    [
        "state_code",
        "state_name",
        "agricultural_land_share"
    ]
]

In [245]:
agricultural_land.shape

(32, 3)

In [246]:
agricultural_land.isna().sum()

state_code                 0
state_name                 0
agricultural_land_share    0
dtype: int64

In [247]:
agricultural_land.head()

,state_code,state_name,agricultural_land_share
0,01,Aguascalientes,55.86
1,02,Baja California,33.23
2,03,Baja California Sur,20.54
3,04,Campeche,55.75
4,05,Coahuila de Zaragoza,44.36


In [248]:
agricultural_land.to_csv(
    "../data/processed/exposure_agricultural_land.csv",
    index=False
)

## 3. Dimension: Sensitivity

### 3.1 Indicator: Multidimensional Poverty

In [30]:
poverty = pd.read_excel(
    "../data/raw/sensitivity/multidimensional_poverty_2022.xlsx",
    sheet_name="Cuadro 4A",
    header=None
)

poverty.head(25)

,0,1,2,3,4,5,6,7,8,9,...,38,39,40,41,42,43,44,45,46,47
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Cuadro 4A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Medición multidimensional de la pobreza*,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Porcentaje, número de personas y carencias pro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,Entidad \nfederativa,Pobreza,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,Porcentaje,NaN,NaN,NaN,NaN,Miles de personas,NaN,...,NaN,Miles de personas,NaN,NaN,NaN,NaN,Carencias promedio,NaN,NaN,NaN
7,NaN,NaN,NaN,2016,2018.000000,2020.000000,2022**,NaN,2016,2018.000,...,NaN,2016,2018.000,2020.000,2022**,NaN,2016,2018.000000,2020.000000,2022**
8,NaN,NaN,Aguascalientes,28.94648,26.265408,27.626693,23.72232,NaN,381.38,360.841,...,NaN,29.204,13.577,34.684,26.095,NaN,3.293076,3.225528,3.424951,3.406783
9,NaN,NaN,Baja California,22.604867,23.598687,22.510117,13.37082,NaN,819.473,884.189,...,NaN,34.839,50.631,58.008,49.912,NaN,3.267574,3.501017,3.352176,3.687991


In [31]:
poverty = pd.read_excel(
    "../data/raw/sensitivity/multidimensional_poverty_2022.xlsx",
    sheet_name="Cuadro 4A",
    header=7
)

In [32]:
poverty.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,2016,2018,2020,2022**,Unnamed: 7,2016.1,2018.1,...,Unnamed: 38,2016.7,2018.7,2020.7,2022**.7,Unnamed: 43,2016.8,2018.8,2020.8,2022**.8
0,NaN,NaN,Aguascalientes,28.946480,26.265408,27.626693,23.722320,NaN,381.380,360.841,...,NaN,29.204,13.577,34.684,26.095,NaN,3.293076,3.225528,3.424951,3.406783
1,NaN,NaN,Baja California,22.604867,23.598687,22.510117,13.370820,NaN,819.473,884.189,...,NaN,34.839,50.631,58.008,49.912,NaN,3.267574,3.501017,3.352176,3.687991
2,NaN,NaN,Baja California Sur,22.880050,18.572164,27.602224,13.329031,NaN,165.234,141.365,...,NaN,10.740,8.887,23.440,6.409,NaN,3.565922,3.724879,3.591937,3.513029
3,NaN,NaN,Campeche,45.660192,48.964296,50.549147,45.129444,NaN,391.464,430.853,...,NaN,56.139,83.375,112.578,91.659,NaN,3.592102,3.580546,3.567944,3.819298
4,NaN,NaN,Coahuila de Zaragoza,27.055713,25.507705,25.617041,18.243322,NaN,799.762,778.060,...,NaN,56.906,46.670,80.982,58.792,NaN,3.457403,3.357317,3.360920,3.502483


In [34]:
poverty = poverty[
    [
        "Unnamed: 2",
        "2022**"
    ]
].copy()

In [35]:
poverty.columns = [
    "state_name",
    "multidimensional_poverty"
]

In [36]:
poverty = poverty.merge(
    state_lookup,
    on="state_name",
    how="left"
)

In [37]:
poverty = poverty[
    [
        "state_code",
        "state_name",
        "multidimensional_poverty"
    ]
]

In [40]:
poverty.head(40)

,state_code,state_name,multidimensional_poverty
0,01,Aguascalientes,23.722320
1,02,Baja California,13.370820
2,03,Baja California Sur,13.329031
3,04,Campeche,45.129444
4,05,Coahuila de Zaragoza,18.243322
5,06,Colima,20.547167
6,07,Chiapas,67.370830
7,08,Chihuahua,17.572199
8,09,Ciudad de México,23.967605
9,10,Durango,34.311067


In [41]:
poverty = poverty.dropna(how="all")

In [44]:
poverty = poverty[
    poverty["state_name"] != "Estados Unidos Mexicanos"
]

In [45]:
poverty.shape

(32, 3)

In [46]:
poverty.columns

Index(['state_code', 'state_name', 'multidimensional_poverty'], dtype='object')

In [47]:
poverty.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32 entries, 0 to 31
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   state_code                32 non-null     object 
 1   state_name                32 non-null     object 
 2   multidimensional_poverty  32 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.0+ KB


In [48]:
poverty.to_csv(
    "../data/processed/sensitivity_multidimensional_poverty.csv",
    index=False
)

### 3.2 Indicator: Marginalization Index

In [287]:
marginalization = pd.read_csv("../data/raw/sensitivity/marginalization_index_2020.csv")

In [288]:
marginalization.head(32)

,CVE_ENT,NOM_ENT,POB_TOT,ANALF,SBASC,OVSDE,OVSEE,OVSAE,OVPT,VHAC,PL_5000,PO2SM,IM_2020,GM_2020,IMN_2020,ENTIDAD,NOM_ENT_etq
0,1,Aguascalientes,1425607,2.111950,23.578264,0.348996,0.231956,0.552683,0.774681,13.132191,21.273184,58.504352,22.205721,Muy bajo,0.816929,Aguascalientes,Aguascalientes
1,2,Baja California,3769020,1.829067,24.681648,0.199850,0.584066,2.099587,1.914523,14.589573,8.455248,73.550861,21.380297,Bajo,0.786563,Baja California,Baja California
2,3,Baja California Sur,798447,2.336256,23.982080,0.419939,0.961081,5.393310,5.059094,18.600851,10.279956,45.486978,21.473388,Bajo,0.789987,Baja California Sur,Baja California Sur
3,4,Campeche,928363,5.863136,29.781429,2.517115,1.045232,3.975752,2.687087,29.973820,29.923855,70.006264,17.805051,Alto,0.655032,Campeche,Campeche
4,5,Coahuila de Zaragoza,3146771,1.672380,21.493970,0.298874,0.169822,0.940492,0.747650,13.475173,10.039625,60.030637,22.545684,Muy bajo,0.829436,Coahuila,Coahuila
5,6,Colima,731391,3.372348,27.818975,0.268863,0.327601,0.662368,2.623618,15.311185,13.496611,59.728189,21.532329,Bajo,0.792156,Colima,Colima
6,7,Chiapas,5543828,13.700680,48.119550,2.458914,1.798601,10.676138,12.388571,36.094395,57.635230,85.570274,11.998654,Muy alto,0.441420,Chiapas,Chiapas
7,8,Chihuahua,3741869,2.630081,27.304743,1.416508,1.660711,1.656981,2.183424,13.599568,14.385886,66.698249,20.015271,Medio,0.736344,Chihuahua,Chihuahua
8,9,Ciudad de México,9209944,1.429621,17.641964,0.051952,0.051820,1.241104,0.633649,14.397804,1.007987,56.132168,23.143109,Muy bajo,0.851415,Ciudad de México,Ciudad de México
9,10,Durango,1832650,2.725664,27.490240,2.844278,2.092870,2.308484,4.257310,16.212389,32.498295,69.255061,18.472740,Alto,0.679596,Durango,Durango


In [289]:
marginalization.columns

Index(['CVE_ENT', 'NOM_ENT', 'POB_TOT', 'ANALF', 'SBASC', 'OVSDE', 'OVSEE',
       'OVSAE', 'OVPT', 'VHAC', 'PL_5000', 'PO2SM', 'IM_2020', 'GM_2020',
       'IMN_2020', 'ENTIDAD', 'NOM_ENT_etq'],
      dtype='object')

In [290]:
marginalization = marginalization[
    [
        "CVE_ENT",
        "NOM_ENT",
        "IM_2020"
    ]
].copy()

In [291]:
marginalization = marginalization.rename(columns={
    "CVE_ENT": "state_code",
    "NOM_ENT": "state_name",
    "IM_2020": "marginalization_index"
})

In [292]:
marginalization["state_code"] = (
    marginalization["state_code"]
    .astype(str)
    .str.zfill(2)
)

In [293]:
marginalization = marginalization.merge(
    state_lookup,
    on="state_code",
    how="left",
    suffixes=("", "_lookup")
)

In [294]:
marginalization["state_name"] = marginalization["state_name_lookup"]

marginalization = marginalization[
    [
        "state_code",
        "state_name",
        "marginalization_index"
    ]
]

In [295]:
marginalization.shape

(32, 3)

In [296]:
marginalization.isna().sum()

state_code               0
state_name               0
marginalization_index    0
dtype: int64

In [61]:
marginalization.to_csv(
    "../data/processed/sensitivity_marginalization.csv",
    index=False
)

### 3.3 Indicator: Population employed in the Primary Sector

In [95]:
primary_sector = pd.read_excel(
    "../data/raw/sensitivity/primary_sector_employed.xls"
)

total_employed = pd.read_excel(
    "../data/raw/sensitivity/total_employed.xls"
)

In [96]:
primary_sector.head(10)

,Población ocupada,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33
0,Sect actividad económica : Primario,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sexo : Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Consulta de: Población ocupada Por: Periodo ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Total,Aguascalientes,Baja California,Baja California Sur,Campeche,Coahuila de Zaragoza,Colima,Chiapas,Chihuahua,...,Quintana Roo,San Luis Potosí,Sinaloa,Sonora,Tabasco,Tamaulipas,Tlaxcala,Veracruz de Ignacio de la Llave,Yucatán,Zacatecas
6,Primer trimestre del 2026,6121224,30905,90049,29156,78076,31247,40826,683999,119382,...,39325,183698,176701,146754,143396,81011,67260,700002,131569,112276
7,Cuarto trimestre del 2025,6227083,30440,83067,27908,72260,38957,45062,659366,115496,...,36908,175683,170841,139912,139724,78003,74742,666454,107685,148583
8,Tercer trimestre del 2025,6469941,27589,92805,33035,68240,44449,44532,719364,130781,...,37706,186417,143911,138743,151584,72950,85855,740616,112249,136572
9,Segundo trimestre del 2025,6296210,26207,94771,30891,63018,52352,40383,719184,130052,...,38934,169873,183516,147840,164747,80018,80331,787557,100184,123683


In [97]:
primary_sector.columns

Index(['Población ocupada', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3',
       'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8',
       'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12',
       'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24',
       'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28',
       'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32',
       'Unnamed: 33'],
      dtype='object')

In [98]:
total_employed.head(10)

,Población ocupada,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33
0,Sect actividad económica : Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sexo : Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Consulta de: Población ocupada Por: Periodo ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Total,Aguascalientes,Baja California,Baja California Sur,Campeche,Coahuila de Zaragoza,Colima,Chiapas,Chihuahua,...,Quintana Roo,San Luis Potosí,Sinaloa,Sonora,Tabasco,Tamaulipas,Tlaxcala,Veracruz de Ignacio de la Llave,Yucatán,Zacatecas
6,Primer trimestre del 2026,59552660,696015,1771951,483495,427100,1603004,392491,2173552,1818648,...,1010964,1281677,1409788,1405467,1105013,1669623,655654,3345424,1242392,652167
7,Cuarto trimestre del 2025,59785854,670177,1749729,473673,423300,1600412,392566,2173811,1856084,...,981364,1264804,1434276,1470027,1098025,1669573,679526,3270790,1249997,705514
8,Tercer trimestre del 2025,59533449,651872,1745730,458459,422752,1514920,379490,2282267,1826410,...,964516,1242963,1419096,1437198,1104202,1651878,693020,3323398,1249993,703871
9,Segundo trimestre del 2025,59440760,652132,1749184,446160,423431,1561461,366562,2220465,1860675,...,987728,1231029,1449072,1462445,1093156,1653268,686868,3398317,1217647,688229


In [99]:
total_employed.columns

Index(['Población ocupada', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3',
       'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8',
       'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12',
       'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24',
       'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28',
       'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32',
       'Unnamed: 33'],
      dtype='object')

In [100]:
def clean_enoe(df):
    # The state names are in row 5
    df.columns = df.iloc[5]

    # Remove metadata rows
    df = df.iloc[6:].reset_index(drop=True)

    # Rename the first column
    df = df.rename(columns={df.columns[0]: "period"})

    return df

In [101]:
primary_sector = clean_enoe(primary_sector)
total_employed = clean_enoe(total_employed)

In [102]:
primary_sector.head()

5,period,Total,Aguascalientes,Baja California,Baja California Sur,Campeche,Coahuila de Zaragoza,Colima,Chiapas,Chihuahua,...,Quintana Roo,San Luis Potosí,Sinaloa,Sonora,Tabasco,Tamaulipas,Tlaxcala,Veracruz de Ignacio de la Llave,Yucatán,Zacatecas
0,Primer trimestre del 2026,6121224,30905,90049,29156,78076,31247,40826,683999,119382,...,39325,183698,176701,146754,143396,81011,67260,700002,131569,112276
1,Cuarto trimestre del 2025,6227083,30440,83067,27908,72260,38957,45062,659366,115496,...,36908,175683,170841,139912,139724,78003,74742,666454,107685,148583
2,Tercer trimestre del 2025,6469941,27589,92805,33035,68240,44449,44532,719364,130781,...,37706,186417,143911,138743,151584,72950,85855,740616,112249,136572
3,Segundo trimestre del 2025,6296210,26207,94771,30891,63018,52352,40383,719184,130052,...,38934,169873,183516,147840,164747,80018,80331,787557,100184,123683
4,Primer trimestre del 2025,6140111,25861,81964,24963,71349,48719,36648,701110,118925,...,30188,165336,181212,130242,170911,80055,75412,811573,89097,102120


In [103]:
primary_sector = primary_sector.dropna(subset=["period"])
total_employed = total_employed.dropna(subset=["period"])

In [106]:
primary_sector = primary_sector[
    primary_sector["period"].astype(str).str.contains("2025", case=False)
]

total_employed = total_employed[
    total_employed["period"].astype(str).str.contains("2025", case=False)
]

In [107]:
primary_sector.tail(10)

5,period,Total,Aguascalientes,Baja California,Baja California Sur,Campeche,Coahuila de Zaragoza,Colima,Chiapas,Chihuahua,...,Quintana Roo,San Luis Potosí,Sinaloa,Sonora,Tabasco,Tamaulipas,Tlaxcala,Veracruz de Ignacio de la Llave,Yucatán,Zacatecas
1,Cuarto trimestre del 2025,6227083,30440,83067,27908,72260,38957,45062,659366,115496,...,36908,175683,170841,139912,139724,78003,74742,666454,107685,148583
2,Tercer trimestre del 2025,6469941,27589,92805,33035,68240,44449,44532,719364,130781,...,37706,186417,143911,138743,151584,72950,85855,740616,112249,136572
3,Segundo trimestre del 2025,6296210,26207,94771,30891,63018,52352,40383,719184,130052,...,38934,169873,183516,147840,164747,80018,80331,787557,100184,123683
4,Primer trimestre del 2025,6140111,25861,81964,24963,71349,48719,36648,701110,118925,...,30188,165336,181212,130242,170911,80055,75412,811573,89097,102120
93,El 27 de mayo de 2025 se reemplazaron los dato...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [108]:
total_employed.tail(10)

5,period,Total,Aguascalientes,Baja California,Baja California Sur,Campeche,Coahuila de Zaragoza,Colima,Chiapas,Chihuahua,...,Quintana Roo,San Luis Potosí,Sinaloa,Sonora,Tabasco,Tamaulipas,Tlaxcala,Veracruz de Ignacio de la Llave,Yucatán,Zacatecas
1,Cuarto trimestre del 2025,59785854,670177,1749729,473673,423300,1600412,392566,2173811,1856084,...,981364,1264804,1434276,1470027,1098025,1669573,679526,3270790,1249997,705514
2,Tercer trimestre del 2025,59533449,651872,1745730,458459,422752,1514920,379490,2282267,1826410,...,964516,1242963,1419096,1437198,1104202,1651878,693020,3323398,1249993,703871
3,Segundo trimestre del 2025,59440760,652132,1749184,446160,423431,1561461,366562,2220465,1860675,...,987728,1231029,1449072,1462445,1093156,1653268,686868,3398317,1217647,688229
4,Primer trimestre del 2025,59001009,668839,1761894,450178,439803,1516383,363903,2252361,1780300,...,971564,1244751,1440348,1388847,1066427,1633145,675389,3417651,1220915,650754
93,El 27 de mayo de 2025 se reemplazaron los dato...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [109]:
primary_sector = primary_sector.drop(index=93)
total_employed = total_employed.drop(index=93)

In [110]:
primary_sector = primary_sector.drop(columns="Total")
total_employed = total_employed.drop(columns="Total")

In [111]:
for col in primary_sector.columns[1:]:
    primary_sector[col] = pd.to_numeric(primary_sector[col])

for col in total_employed.columns[1:]:
    total_employed[col] = pd.to_numeric(total_employed[col])

In [112]:
primary_avg = primary_sector.drop(columns="period").mean()

total_avg = total_employed.drop(columns="period").mean()

In [113]:
employment = pd.DataFrame({
    "state_name": primary_avg.index,
    "primary_employed": primary_avg.values,
    "total_employed": total_avg.values
})

In [114]:
employment.head()

,state_name,primary_employed,total_employed
0,Aguascalientes,27524.25,660755.00
1,Baja California,88151.75,1751634.25
2,Baja California Sur,29199.25,457117.50
3,Campeche,68716.75,427321.50
4,Coahuila de Zaragoza,46119.25,1548294.00


In [115]:
employment["primary_sector_employment"] = (
    employment["primary_employed"] /
    employment["total_employed"]
) * 100

In [116]:
employment["primary_sector_employment"] = (
    employment["primary_sector_employment"]
    .round(2)
)

In [117]:
employment_primary_sector = employment[
    [
        "state_name",
        "primary_sector_employment"
    ]
]

In [118]:
employment_primary_sector = employment_primary_sector.merge(
    state_lookup,
    on="state_name",
    how="left")

In [119]:
employment_primary_sector = employment_primary_sector[
    [
        "state_code",
        "state_name",
        "primary_sector_employment"
    ]
]

In [120]:
employment_primary_sector.head()

,state_code,state_name,primary_sector_employment
0,01,Aguascalientes,4.17
1,02,Baja California,5.03
2,03,Baja California Sur,6.39
3,04,Campeche,16.08
4,05,Coahuila de Zaragoza,2.98


In [123]:
employment_primary_sector.shape

(32, 3)

In [124]:
employment_primary_sector.describe()

,primary_sector_employment
count,32.000000
mean,11.525313
std,7.570266
min,0.450000
25%,5.090000
50%,10.530000
75%,16.227500
max,31.350000


In [125]:
employment_primary_sector.to_csv(
    "../data/processed/sensitivity_primary_sector_employment.csv",
    index=False
)

## 4. Dimension: Adaptive Capacity

### 4.1 Indicator: Educational Attainment

In [144]:
educational_attainment = pd.read_excel(
    "../data/raw/adaptive capacity/education.xlsx"
)

In [145]:
educational_attainment.head(20)

,Instituto Nacional de Estadística y Geografía (INEGI),Unnamed: 1
0,Grado promedio de escolaridad de la población ...,NaN
1,NaN,NaN
2,Fecha de consulta: 05/08/2026 07:37:29,NaN
3,NaN,NaN
4,Entidad federativa,2020
5,Entidad federativa,Total
6,Estados Unidos Mexicanos,9.74
7,Aguascalientes,10.35
8,Baja California,10.2
9,Baja California Sur,10.34


In [146]:
educational_attainment.columns = ["state_name", "educational_attainment"]

In [147]:
educational_attainment = educational_attainment.iloc[6:].reset_index(drop=True)

In [148]:
educational_attainment = educational_attainment[
    educational_attainment["state_name"] != "Estados Unidos Mexicanos"
].reset_index(drop=True)

In [152]:
educational_attainment.head(20)

,state_name,educational_attainment
0,Aguascalientes,10.35
1,Baja California,10.2
2,Baja California Sur,10.34
3,Campeche,9.63
4,Coahuila de Zaragoza,10.43
5,Colima,10.05
6,Chiapas,7.78
7,Chihuahua,10
8,Ciudad de México,11.48
9,Durango,9.75


In [150]:
educational_attainment = educational_attainment[
    educational_attainment["educational_attainment"].notna()
].reset_index(drop=True)

In [153]:
educational_attainment["educational_attainment"] = pd.to_numeric(
    educational_attainment["educational_attainment"]
)

In [154]:
educational_attainment = educational_attainment.merge(
    state_lookup,
    on="state_name",
    how="left"
)

In [155]:
educational_attainment = educational_attainment[
    [
        "state_code",
        "state_name",
        "educational_attainment"
    ]
]

In [158]:
educational_attainment.shape

(32, 3)

In [159]:
educational_attainment.to_csv(
    "../data/processed/adaptive_capacity_educational_attainment.csv",
    index=False
)

In [160]:
educational_attainment.head()

,state_code,state_name,educational_attainment
0,01,Aguascalientes,10.35
1,02,Baja California,10.20
2,03,Baja California Sur,10.34
3,04,Campeche,9.63
4,05,Coahuila de Zaragoza,10.43


### 4.2 and 4.3 Indicators: Access to Drainage and Piped Water

In [164]:
water_services = pd.read_excel(
    "../data/raw/adaptive capacity/basic_water_services.xlsx"
)

In [165]:
water_services.head(10)

,Instituto Nacional de Estadística y Geografía (INEGI),Unnamed: 1,Unnamed: 2
0,Viviendas particulares habitadas por entidad f...,NaN,NaN
1,NaN,NaN,NaN
2,Fecha de consulta: 05/08/2026 09:23:11,NaN,NaN
3,NaN,NaN,NaN
4,Entidad federativa,Disponibilidad de servicios,2020.0
5,Aguascalientes,Total viviendas particulares habitadas,386011.0
6,Aguascalientes,Disponen de agua entubada dentro de la vivienda,367744.0
7,Aguascalientes,Disponen de drenaje,383148.0
8,Baja California,Total viviendas particulares habitadas,1144251.0
9,Baja California,Disponen de agua entubada dentro de la vivienda,1063106.0


In [166]:
water_services.columns = [
    "state_name",
    "service",
    "value"
]

In [167]:
water_services = water_services.iloc[5:].reset_index(drop=True)

In [168]:
water_services["value"] = pd.to_numeric(
    water_services["value"],
    errors="coerce"
)

In [169]:
water_services = water_services.dropna(subset=["value"])

In [171]:
water_services.tail()

,state_name,service,value
91,Yucatán,Disponen de agua entubada dentro de la vivienda,513253.0
92,Yucatán,Disponen de drenaje,605627.0
93,Zacatecas,Total viviendas particulares habitadas,442263.0
94,Zacatecas,Disponen de agua entubada dentro de la vivienda,342717.0
95,Zacatecas,Disponen de drenaje,425727.0


In [172]:
water_services = water_services.pivot(
    index="state_name",
    columns="service",
    values="value"
).reset_index()

In [173]:
water_services.head()

service,state_name,Disponen de agua entubada dentro de la vivienda,Disponen de drenaje,Total viviendas particulares habitadas
0,Aguascalientes,367744.0,383148.0,386011.0
1,Baja California,1063106.0,1101573.0,1144251.0
2,Baja California Sur,196970.0,231864.0,239358.0
3,Campeche,168344.0,245656.0,260221.0
4,Chiapas,705992.0,1226871.0,1348105.0


In [174]:
water_services = water_services.rename(columns={
    "Total viviendas particulares habitadas": "total_dwellings",
    "Disponen de agua entubada dentro de la vivienda": "piped_water",
    "Disponen de drenaje": "drainage"
})

In [175]:
water_services["access_to_piped_water"] = (
    water_services["piped_water"] /
    water_services["total_dwellings"]
) * 100

water_services["access_to_drainage"] = (
    water_services["drainage"] /
    water_services["total_dwellings"]
) * 100

In [176]:
water_services["access_to_piped_water"] = (
    water_services["access_to_piped_water"].round(2)
)

water_services["access_to_drainage"] = (
    water_services["access_to_drainage"].round(2)
)

In [177]:
water_services = water_services.merge(
    state_lookup,
    on="state_name",
    how="left"
)

In [178]:
water_services = water_services[
    [
        "state_code",
        "state_name",
        "access_to_piped_water",
        "access_to_drainage"
    ]
]

In [179]:
water_services.shape

(32, 4)

In [180]:
water_services.isna().sum()

state_code               0
state_name               0
access_to_piped_water    0
access_to_drainage       0
dtype: int64

In [182]:
water_services.describe()

,access_to_piped_water,access_to_drainage
count,32.000000,32.000000
mean,77.561250,95.344062
std,14.615586,3.889490
min,40.620000,80.340000
25%,66.877500,94.745000
50%,79.085000,96.390000
75%,89.720000,97.387500
max,97.070000,99.710000


In [183]:
water_services.to_csv(
    "../data/processed/adaptive_capacity_basic_services.csv",
    index=False
)

### 4.4 Indicator: Population affiliated with health services (%)

In [184]:
health = pd.read_excel(
    "../data/raw/adaptive capacity/health_access.xlsx"
)

In [186]:
health.head(10)

,Instituto Nacional de Estadística y Geografía (INEGI),Unnamed: 1,Unnamed: 2
0,Población según condición de afiliación a serv...,NaN,NaN
1,NaN,NaN,NaN
2,Fecha de consulta: 05/08/2026 09:40:43,NaN,NaN
3,NaN,NaN,NaN
4,Entidad federativa,Total,Total
5,Entidad federativa,Total,Afiliada
6,Aguascalientes,1425607,1161139
7,Baja California,3769020,2905265
8,Baja California Sur,798447,664122
9,Campeche,928363,719677


In [187]:
health.columns = [
    "state_name",
    "total_population",
    "affiliated_population"
]

In [188]:
health = health.iloc[6:].reset_index(drop=True)

In [189]:
health.head()

,state_name,total_population,affiliated_population
0,Aguascalientes,1425607,1161139
1,Baja California,3769020,2905265
2,Baja California Sur,798447,664122
3,Campeche,928363,719677
4,Coahuila de Zaragoza,3146771,2540708


In [190]:
health["total_population"] = pd.to_numeric(
    health["total_population"],
    errors="coerce"
)

health["affiliated_population"] = pd.to_numeric(
    health["affiliated_population"],
    errors="coerce"
)

In [191]:
health = health.dropna(subset=["total_population"])

In [192]:
health.tail()

,state_name,total_population,affiliated_population
27,Tamaulipas,3527735.0,2803407.0
28,Tlaxcala,1342977.0,964599.0
29,Veracruz de Ignacio de la Llave,8062579.0,5825533.0
30,Yucatán,2320898.0,1810121.0
31,Zacatecas,1622138.0,1293059.0


In [193]:
health["health_service_affiliation"] = (
    health["affiliated_population"] /
    health["total_population"]
) * 100

In [194]:
health["health_service_affiliation"] = (
    health["health_service_affiliation"]
    .round(2)
)

In [195]:
health = health.merge(
    state_lookup,
    on="state_name",
    how="left"
)

In [196]:
health = health[
    [
        "state_code",
        "state_name",
        "health_service_affiliation"
    ]
]

In [197]:
health.shape

(32, 3)

In [198]:
health.isna().sum()

state_code                    0
state_name                    0
health_service_affiliation    0
dtype: int64

In [199]:
health.describe()

,health_service_affiliation
count,32.000000
mean,75.655000
std,5.747437
min,62.220000
25%,71.535000
50%,77.300000
75%,80.770000
max,84.350000


In [200]:
health.to_csv(
    "../data/processed/adaptive_capacity_health_service_affiliation.csv",
    index=False
)